# 多输入多输出通道
:label:`sec_channels`

虽然我们在 :numref:`subsec_why-conv-channels`中描述了构成每个图像的多个通道和多层卷积层。例如彩色图像具有标准的RGB通道来代表红、绿和蓝。
但是到目前为止，我们仅展示了单个输入和单个输出通道的简化例子。
这使得我们可以将输入、卷积核和输出看作二维张量。

当我们添加通道时，我们的输入和隐藏的表示都变成了三维张量。例如，每个RGB输入图像具有$3\times h\times w$的形状。我们将这个大小为$3$的轴称为*通道*（channel）维度。本节将更深入地研究具有多输入和多输出通道的卷积核。

## 多输入通道

当输入包含多个通道时，需要构造一个与输入数据具有相同输入通道数的卷积核，以便与输入数据进行互相关运算。假设输入的通道数为$c_i$，那么卷积核的输入通道数也需要为$c_i$。如果卷积核的窗口形状是$k_h\times k_w$，那么当$c_i=1$时，我们可以把卷积核看作形状为$k_h\times k_w$的二维张量。

然而，当$c_i>1$时，我们卷积核的每个输入通道将包含形状为$k_h\times k_w$的张量。将这些张量$c_i$连结在一起可以得到形状为$c_i\times k_h\times k_w$的卷积核。**由于输入和卷积核都有$c_i$个通道，我们可以对每个通道输入的二维张量和卷积核的二维张量进行互相关运算，再对通道求和（将$c_i$的结果相加）得到二维张量。**这是多通道输入和多输入通道卷积核之间进行二维互相关运算的结果。

在 :numref:`fig_conv_multi_in`中，我们演示了一个具有两个输入通道的二维互相关运算的示例。阴影部分是第一个输出元素以及用于计算这个输出的输入和核张量元素：$(1\times1+2\times2+4\times3+5\times4)+(0\times0+1\times1+3\times2+4\times3)=56$。

![两个输入通道的互相关计算。](../img/conv-multi-in.svg)
:label:`fig_conv_multi_in`

为了加深理解，我们(**实现一下多输入通道互相关运算**)。
简而言之，我们所做的就是对每个通道执行互相关操作，然后将结果相加。


In [1]:
import torch
from d2l import torch as d2l

In [2]:
def corr2d_multi_in(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

我们可以构造与 :numref:`fig_conv_multi_in`中的值相对应的输入张量`X`和核张量`K`，以(**验证互相关运算的输出**)。


In [3]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

## 多输出通道

到目前为止，不论有多少输入通道，我们还只有一个输出通道。然而，正如我们在 :numref:`subsec_why-conv-channels`中所讨论的，每一层有多个输出通道是至关重要的。在最流行的神经网络架构中，随着神经网络层数的加深，我们常会增加输出通道的维数，通过减少空间分辨率以获得更大的通道深度。直观地说，我们可以将每个通道看作对不同特征的响应。而现实可能更为复杂一些，因为每个通道不是独立学习的，而是为了共同使用而优化的。因此，多输出通道并不仅是学习多个单通道的检测器。

用$c_i$和$c_o$分别表示输入和输出通道的数目，并让$k_h$和$k_w$为卷积核的高度和宽度。为了获得多个通道的输出，我们可以为每个输出通道创建一个形状为$c_i\times k_h\times k_w$的卷积核张量，这样卷积核的形状是$c_o\times c_i\times k_h\times k_w$。在互相关运算中，每个输出通道先获取所有输入通道，再以对应该输出通道的卷积核计算出结果。

如下所示，我们实现一个[**计算多个通道的输出的互相关函数**]。


In [4]:
def corr2d_multi_in_out(X, K):
    # 迭代“K”的第0个维度，每次都对输入“X”执行互相关运算。
    # 最后将所有结果都叠加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

#### 补充：torch.stack 函数详解

<small>上面的 `corr2d_multi_in_out` 中使用了 `torch.stack`，这里补充介绍一下它的用法。</small>

**torch.stack 的作用**

<small>`torch.stack(tensors, dim=0)` 会**沿着一个新插入的维度**把一组张量堆叠起来，即**新增一个轴**。</small>

- <small>传入的所有张量形状必须**完全一致**。</small>
- <small>若传入 $n$ 个形状均为 $(a, b, c)$ 的张量，用 `dim=0` 堆叠后形状变为 $(n, a, b, c)$。</small>
- <small>返回的新张量多出一个维度，其大小等于传入张量的个数 $n$。</small>

**与 torch.cat 的区别**

- <small>`torch.cat`：沿**已有的某个维度**进行拼接，**不会新增维度**，只要求除拼接维度外其余维度形状一致。</small>
- <small>`torch.stack`：**新增**一个维度并沿它堆叠，要求所有张量形状完全一致。</small>

下面的代码 cell 用一个简单例子对比 `torch.stack` 与 `torch.cat` 的输出形状。

**回到本节的代码**

<small>在 `corr2d_multi_in_out` 中，列表推导式 `[corr2d_multi_in(X, k) for k in K]` 对 $c_o$ 个输出通道分别得到形状为 $(h', w')$ 的二维结果；随后 `torch.stack(..., 0)` 沿**第 0 维新增一个通道轴**，把它们堆叠成形状为 $(c_o, h', w')$ 的三维输出张量——这正是"多输出通道"这一维度的由来。</small>


In [6]:
import torch

a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print(torch.stack([a, b], dim=1))
print(torch.stack([a, b], dim=0))   # torch.Size([2, 2, 2])
# stack：新增第 0 维 -> 形状 (2, 2, 2)
print(torch.stack([a, b], dim=0).shape)   # torch.Size([2, 2, 2])

# cat：沿已有第 0 维拼接，不新增维度 -> 形状 (4, 2)
print(torch.cat([a, b], dim=0).shape)     # torch.Size([4, 2])


tensor([[[1., 2.],
         [5., 6.]],

        [[3., 4.],
         [7., 8.]]])
tensor([[[1., 2.],
         [3., 4.]],

        [[5., 6.],
         [7., 8.]]])
torch.Size([2, 2, 2])
torch.Size([4, 2])


通过将核张量`K`与`K+1`（`K`中每个元素加$1$）和`K+2`连接起来，构造了一个具有$3$个输出通道的卷积核。


In [7]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [8]:
K

tensor([[[[0., 1.],
          [2., 3.]],

         [[1., 2.],
          [3., 4.]]],


        [[[1., 2.],
          [3., 4.]],

         [[2., 3.],
          [4., 5.]]],


        [[[2., 3.],
          [4., 5.]],

         [[3., 4.],
          [5., 6.]]]])

下面，我们对输入张量`X`与卷积核张量`K`执行互相关运算。现在的输出包含$3$个通道，第一个通道的结果与先前输入张量`X`和多输入单输出通道的结果一致。


In [9]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

## $1\times 1$ 卷积层

[~~1x1卷积~~]

$1 \times 1$卷积，即$k_h = k_w = 1$，看起来似乎没有多大意义。
毕竟，卷积的本质是有效提取相邻像素间的相关特征，而$1 \times 1$卷积显然没有此作用。
尽管如此，$1 \times 1$仍然十分流行，经常包含在复杂深层网络的设计中。下面，让我们详细地解读一下它的实际作用。

因为使用了最小窗口，$1\times 1$卷积失去了卷积层的特有能力——在高度和宽度维度上，识别相邻元素间相互作用的能力。
其实$1\times 1$卷积的唯一计算发生在通道上。

 :numref:`fig_conv_1x1`展示了使用$1\times 1$卷积核与$3$个输入通道和$2$个输出通道的互相关计算。
这里输入和输出具有相同的高度和宽度，输出中的每个元素都是从输入图像中同一位置的元素的线性组合。
我们可以将$1\times 1$卷积层看作在每个像素位置应用的全连接层，以$c_i$个输入值转换为$c_o$个输出值。
因为这仍然是一个卷积层，所以跨像素的权重是一致的。
同时，$1\times 1$卷积层需要的权重维度为$c_o\times c_i$，再额外加上一个偏置。

![互相关计算使用了具有3个输入通道和2个输出通道的 $1\times 1$ 卷积核。其中，输入和输出具有相同的高度和宽度。](../img/conv-1x1.svg)
:label:`fig_conv_1x1`

下面，我们使用全连接层实现$1 \times 1$卷积。
请注意，我们需要对输入和输出的数据形状进行调整。


In [11]:
def corr2d_multi_in_out_1x1(X, K):
    # 用“全连接层/矩阵乘法”实现 1x1 卷积
    # X 形状: (c_i, h, w)      K 形状: (c_o, c_i, 1, 1)

    # 解包 X 的形状：输入通道数 c_i、高 h、宽 w
    c_i, h, w = X.shape
    # 输出通道数 c_o，即卷积核 K 第 0 维的长度
    c_o = K.shape[0]
    # 把 X 从 (c_i, h, w) 展平成 (c_i, h*w)：
    # 每个通道的空间像素被“拉直”成一维，方便一次对所有像素位置做矩阵乘法
    X = X.reshape((c_i, h * w))
    # 把 K 从 (c_o, c_i, 1, 1) 展平成 (c_o, c_i)：
    # 1x1 卷积核没有空间尺寸，只保留“输入通道->输出通道”的权重，即全连接层的权重矩阵
    K = K.reshape((c_o, c_i))
    # 全连接层中的矩阵乘法：Y[c_o, h*w] = K[c_o, c_i] @ X[c_i, h*w]
    # 每个输出像素 = 同一位置所有输入通道的线性加权组合（这正是 1x1 卷积的本质）
    Y = torch.matmul(K, X)
    # 把结果从 (c_o, h*w) 重新变回 (c_o, h, w)，高和宽保持不变
    return Y.reshape((c_o, h, w))


为什么等价于普通卷积？ 因为 1×1 卷积窗口不覆盖相邻像素，每个输出像素只与同一位置的所有输入通道有关——这就是逐像素位置上的全连接层。而普通卷积（窗口 >1）还要在空间邻域上滑动求和，无法直接拆成这种纯矩阵乘法。

当执行$1\times 1$卷积运算时，上述函数相当于先前实现的互相关函数`corr2d_multi_in_out`。让我们用一些样本数据来验证这一点。


In [17]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

In [19]:
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

#### 补充：assert 断言的作用与用法

<small>上面代码的最后一行用到了 `assert`，这里补充解释一下。</small>

**assert 是什么**

<small>`assert` 是 Python 的**断言语句**，用于在代码里做"条件检查"：如果条件为真就静默通过、继续执行；如果条件为假就抛出 `AssertionError` 并中断程序。常用来自动验证某个假设是否成立（类似单元测试）。</small>

<small>语法：`assert 条件表达式, "出错时显示的信息"`</small>

**在这个 notebook 中的含义**

<small>`assert float(torch.abs(Y1 - Y2).sum()) < 1e-6` 就是验证：两种方式算出的 `Y1`、`Y2` 之间总误差足够小（< 1e-6，视为相等）。若两种 $1\times1$ 卷积实现不一致，这里会立刻报错。</small>

**使用 assert 的注意点**

- <small>条件为真 → 什么都不发生；条件为假 → 抛 `AssertionError`。</small>
- <small>第二个参数（可选）是报错时显示的消息，便于排查。</small>
- <small>可用 `python -O` 运行时，`assert` 会被**跳过**，所以它适合调试/校验，不能代替正式的输入校验或错误处理（那些应使用 `if` + `raise`）。</small>

**断言失败的报错示例**

<small>下面的代码 cell 演示了断言通过和失败两种情形——请先运行一次看看效果（失败那条会抛出 `AssertionError`，属正常现象）。</small>


In [20]:
# 演示 1：断言通过（条件为真，什么都不发生，继续往下执行）
a = 1
b = 1
assert a == b, "a 和 b 不相等"
print("断言通过：a == b，程序继续运行")

# 演示 2：条件为假会抛出 AssertionError
# 这里用 try/except 捕获它，避免中断整个 notebook 的运行
try:
    x = 1
    y = 2
    assert x == y, "x 和 y 不相等"
except AssertionError as e:
    print("断言失败，捕获到的错误信息：", e)


断言通过：a == b，程序继续运行
断言失败，捕获到的错误信息： x 和 y 不相等


## 小结

* 多输入多输出通道可以用来扩展卷积层的模型。
* 当以每像素为基础应用时，$1\times 1$卷积层相当于全连接层。
* $1\times 1$卷积层通常用于调整网络层的通道数量和控制模型复杂性。

## 练习

1. 假设我们有两个卷积核，大小分别为$k_1$和$k_2$（中间没有非线性激活函数）。
    1. 证明运算可以用单次卷积来表示。
    1. 这个等效的单个卷积核的维数是多少呢？
    1. 反之亦然吗？
1. 假设输入为$c_i\times h\times w$，卷积核大小为$c_o\times c_i\times k_h\times k_w$，填充为$(p_h, p_w)$，步幅为$(s_h, s_w)$。
    1. 前向传播的计算成本（乘法和加法）是多少？
    1. 内存占用是多少？
    1. 反向传播的内存占用是多少？
    1. 反向传播的计算成本是多少？
1. 如果我们将输入通道$c_i$和输出通道$c_o$的数量加倍，计算数量会增加多少？如果我们把填充数量翻一番会怎么样？
1. 如果卷积核的高度和宽度是$k_h=k_w=1$，前向传播的计算复杂度是多少？
1. 本节最后一个示例中的变量`Y1`和`Y2`是否完全相同？为什么？
1. 当卷积窗口不是$1\times 1$时，如何使用矩阵乘法实现卷积？


[Discussions](https://discuss.d2l.ai/t/1854)
